# Paper Results: Survey x Simulation

This notebook mixes the two sides of the project:

- `survey_results.ipynb` explains and explores the human survey data.
- `simulation_results.ipynb` explains and explores the logged simulation benchmark data.
- This file checks whether the simulation benchmark agrees with the survey outcomes.

The main question here is not whether the scores are numerically identical. The survey is a human rating on a `-3..+3` Likert-style scale, while the simulation benchmark is a normalized objective score on `0..1`. The question is whether they tell the same story when we compare the same policy/POV pairs.

## Decision Logic

We will treat this as an alignment check:

1. Import and clean the survey answers.
2. Import and aggregate the simulation metrics.
3. Put both datasets on comparable condition/axis labels.
4. Compare rankings and paired differences, not raw absolute scores.
5. Mark where the simulation and survey agree, disagree, or are too close to call.

Useful references for the methods used here:

- Exploratory data analysis: [NIST/SEMATECH EDA handbook](https://www.nist.gov/publications/nistsematech-e-handbook-statistical-methods-chapter-1-exploratory-data-analysis)
- Bootstrap confidence intervals: [SciPy `stats.bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html)
- Rank correlation / Spearman idea: [SciPy `stats.spearmanr`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html)

We implement the bootstrap and rank correlation directly with `numpy`/`pandas` so this notebook does not require SciPy.

## Imports And Shared Mappings

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "analysis" else Path("paper") / "analysis"
PAPER_DIR = NOTEBOOK_DIR.parent
DATA_DIR = PAPER_DIR / "data" / "human-survey"
RESULTS_DIR = PAPER_DIR / "results"
GENERATED_DIR = PAPER_DIR / "generated"

SURVEY_CSV = DATA_DIR / "responses.csv"
SIMULATION_ROOT = PAPER_DIR.parent / "artifacts" / "local-validation"

RANDOM_SEED = 20260514


In [ ]:
Q_MAPPING = {
    "q1": {"policy": "A", "pov": "first_person", "condition_label": "Policy A - first person"},
    "q2": {"policy": "B", "pov": "first_person", "condition_label": "Policy B - first person"},
    "q3": {"policy": "A", "pov": "third_person", "condition_label": "Policy A - third person"},
    "q4": {"policy": "B", "pov": "third_person", "condition_label": "Policy B - third person"},
}

SURVEY_QUESTION_AXIS = {
    "1": "competence_control",
    "2": "safety",
    "3": "awareness",
    "4": "positive_impression",
}

SIMULATION_AXIS_MAP = {
    "perceived_dexterity": "competence_control",
    "perceived_safety": "safety",
    "perceived_social_awareness": "awareness",
    "impression": "positive_impression",
}

EPISODE_LABELS = {
    "ep1": "no_obstacles",
    "ep2": "fixed_obstacles",
    "ep3": "moving_people",
}

AXES = ["competence_control", "safety", "awareness", "positive_impression"]
AXES_WITH_GLOBAL = ["global"] + AXES

COMPARISONS = [
    ("q1", "q2", "A vs B - first-person POV", "Policy A minus Policy B, first-person videos"),
    ("q3", "q4", "A vs B - third-person POV", "Policy A minus Policy B, third-person videos"),
    ("q1", "q3", "First vs third POV - Policy A", "First-person minus third-person for Policy A"),
    ("q2", "q4", "First vs third POV - Policy B", "First-person minus third-person for Policy B"),
]

## Import Survey Results

The survey data is participant-level. Every participant saw all four videos, so policy and POV comparisons are paired within participant. We keep that paired structure when computing differences.

In [ ]:
def convert_answer_to_number(answer):
    answer_text = str(answer)
    for score_text, score_number in {"-3": -3, "-2": -2, "-1": -1, "0": 0, "+1": 1, "+2": 2, "+3": 3}.items():
        if answer_text.startswith(score_text):
            return score_number
    return np.nan


def familiarity_to_boolean(answer):
    answer_text = str(answer)
    if answer_text == "True" or answer_text.startswith("Daily interaction"):
        return True
    if answer_text == "False" or answer_text.startswith("No daily contact"):
        return False
    return pd.NA


raw_survey = pd.read_csv(SURVEY_CSV)

rename_columns = {}
for old_column_name in raw_survey.columns:
    if old_column_name == "Timestamp":
        rename_columns[old_column_name] = "timestamp"
    elif old_column_name.startswith("Background / Familiarity"):
        rename_columns[old_column_name] = "robotics_familiarity"
    elif old_column_name.startswith("Q"):
        question_code = old_column_name.split(" ", 1)[0]
        rename_columns[old_column_name] = question_code.lower().replace(".", "_")

survey_wide = raw_survey.rename(columns=rename_columns).copy()
survey_wide.insert(0, "participant_id", np.arange(1, len(survey_wide) + 1))

score_columns = [column for column in survey_wide.columns if re.fullmatch(r"q[1-4]_[1-4]", column)]
for column in score_columns:
    survey_wide[column] = survey_wide[column].apply(convert_answer_to_number)

survey_wide["robotics_familiarity"] = survey_wide["robotics_familiarity"].apply(familiarity_to_boolean).astype("boolean")

survey_rows = []
for condition, metadata in Q_MAPPING.items():
    for question_number, axis in SURVEY_QUESTION_AXIS.items():
        column = f"{condition}_{question_number}"
        if column not in survey_wide.columns:
            continue
        for _, row in survey_wide.iterrows():
            survey_rows.append(
                {
                    "participant_id": row["participant_id"],
                    "robotics_familiarity": row["robotics_familiarity"],
                    "condition": condition,
                    "condition_label": metadata["condition_label"],
                    "policy": metadata["policy"],
                    "pov": metadata["pov"],
                    "axis": axis,
                    "score": row[column],
                }
            )

survey_long = pd.DataFrame(survey_rows)
survey_long["score_norm"] = (survey_long["score"] + 3) / 6

survey_global = (
    survey_long.groupby(["participant_id", "robotics_familiarity", "condition", "condition_label", "policy", "pov"], dropna=False)
    .agg(score=("score", "mean"), score_norm=("score_norm", "mean"))
    .reset_index()
)
survey_global["axis"] = "global"

survey_analysis_long = pd.concat([survey_long, survey_global], ignore_index=True, sort=False)

print(f"Survey participants: {survey_wide['participant_id'].nunique()}")
print(f"Survey long rows including global: {len(survey_analysis_long)}")
display(survey_analysis_long.head())

In [ ]:
survey_keys = ["condition", "condition_label", "policy", "pov", "axis"]

survey_summary = (
    survey_analysis_long.groupby(survey_keys, dropna=False)
    .agg(
        n=("score", "count"),
        survey_mean_raw=("score", "mean"),
        survey_median_raw=("score", "median"),
        survey_mean_0_1=("score_norm", "mean"),
        survey_median_0_1=("score_norm", "median"),
        survey_std_0_1=("score_norm", "std"),
    )
    .reset_index()
)

survey_polarity = (
    survey_analysis_long.assign(
        negative=survey_analysis_long["score"] < 0,
        neutral=survey_analysis_long["score"] == 0,
        positive=survey_analysis_long["score"] > 0,
    )
    .groupby(survey_keys, dropna=False)[["negative", "neutral", "positive"]]
    .mean()
    .mul(100)
    .reset_index()
    .rename(columns={"negative": "percent_negative", "neutral": "percent_neutral", "positive": "percent_positive"})
)

survey_summary = survey_summary.merge(survey_polarity, on=survey_keys, how="left")

display(survey_summary.sort_values(["axis", "condition"]))

## Import Simulation Results

The simulation data is episode-level. It has repeated logged runs, not human participants. We aggregate repeated runs within each `condition x episode x axis`, then summarize across the three episode slots.

In [ ]:
def metadata_from_csv_path(path: Path) -> dict:
    relative_parts = path.relative_to(SIMULATION_ROOT).parts
    condition = relative_parts[0] if relative_parts else None
    run_label = relative_parts[1] if len(relative_parts) > 1 else path.parent.name
    episode_match = re.search(r"ep(\d+)", run_label)
    episode_slot = f"ep{episode_match.group(1)}" if episode_match else None
    metadata = Q_MAPPING.get(condition, {})
    return {
        "condition": condition,
        "condition_label": metadata.get("condition_label"),
        "policy": metadata.get("policy"),
        "pov": metadata.get("pov"),
        "episode_slot": episode_slot,
        "episode_type": EPISODE_LABELS.get(episode_slot),
        "run_label": run_label,
        "csv_path": str(path),
    }


simulation_csv_paths = sorted(SIMULATION_ROOT.rglob("episode-metrics-*.csv"))
if not simulation_csv_paths:
    raise FileNotFoundError(f"No simulation CSV files found under {SIMULATION_ROOT.resolve()}")

simulation_frames = []
for csv_path in simulation_csv_paths:
    frame = pd.read_csv(csv_path)
    for key, value in metadata_from_csv_path(csv_path).items():
        frame[key] = value
    simulation_frames.append(frame)

simulation_raw = pd.concat(simulation_frames, ignore_index=True)

for column in SIMULATION_AXIS_MAP:
    simulation_raw[column] = pd.to_numeric(simulation_raw[column], errors="coerce")

if "technical_valid" in simulation_raw.columns:
    simulation_raw["technical_valid_bool"] = simulation_raw["technical_valid"].astype(str).str.lower().eq("true")

simulation_episode_axes = (
    simulation_raw.groupby(["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"], dropna=False)[list(SIMULATION_AXIS_MAP)]
    .mean()
    .reset_index()
    .rename(columns=SIMULATION_AXIS_MAP)
)

simulation_axis_long = simulation_episode_axes.melt(
    id_vars=["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"],
    value_vars=AXES,
    var_name="axis",
    value_name="sim_score",
).dropna(subset=["sim_score"])

simulation_global = (
    simulation_axis_long.groupby(["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"], dropna=False)
    .agg(sim_score=("sim_score", "mean"))
    .reset_index()
)
simulation_global["axis"] = "global"

simulation_analysis_long = pd.concat([simulation_axis_long, simulation_global], ignore_index=True, sort=False)

print(f"Simulation rows: {len(simulation_raw)}")
print(f"Simulation CSV files: {len(simulation_csv_paths)}")
display(simulation_analysis_long.head())

In [ ]:
simulation_keys = ["condition", "condition_label", "policy", "pov", "axis"]

simulation_summary = (
    simulation_analysis_long.groupby(simulation_keys, dropna=False)
    .agg(
        sim_mean_0_1=("sim_score", "mean"),
        sim_median_0_1=("sim_score", "median"),
        sim_std_0_1=("sim_score", "std"),
        n_episode_values=("sim_score", "count"),
        n_episode_slots=("episode_slot", "nunique"),
    )
    .reset_index()
)

simulation_run_quality = (
    simulation_raw.groupby(["condition", "condition_label", "policy", "pov"], dropna=False)
    .agg(
        n_logged_runs=("csv_path", "count"),
        technical_valid_rate=("technical_valid_bool", "mean") if "technical_valid_bool" in simulation_raw.columns else ("csv_path", "count"),
    )
    .reset_index()
)

simulation_summary = simulation_summary.merge(simulation_run_quality, on=["condition", "condition_label", "policy", "pov"], how="left")

display(simulation_summary.sort_values(["axis", "condition"]))

## Put Survey And Simulation On The Same Table

The survey score is converted to `0..1` only for comparison. A value of `0.5` means neutral on the original `-3..+3` scale. The simulation score already lives on `0..1`.

In [ ]:
joint_summary = survey_summary.merge(
    simulation_summary,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="outer",
)

joint_summary["mean_gap_sim_minus_survey"] = joint_summary["sim_mean_0_1"] - joint_summary["survey_mean_0_1"]

display(
    joint_summary[
        [
            "condition",
            "condition_label",
            "axis",
            "survey_mean_raw",
            "survey_mean_0_1",
            "sim_mean_0_1",
            "mean_gap_sim_minus_survey",
            "n",
            "n_episode_slots",
            "n_logged_runs",
            "technical_valid_rate",
        ]
    ].sort_values(["axis", "condition"])
)

In [ ]:
plot_data = joint_summary.dropna(subset=["survey_mean_0_1", "sim_mean_0_1"]).copy()

fig, ax = plt.subplots(figsize=(7, 7))
for axis_name, axis_frame in plot_data.groupby("axis"):
    ax.scatter(axis_frame["survey_mean_0_1"], axis_frame["sim_mean_0_1"], label=axis_name, s=70)

ax.axline((0, 0), slope=1, color="black", linewidth=1, linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Survey mean, normalized to 0..1")
ax.set_ylabel("Simulation mean, 0..1")
ax.set_title("Absolute score check: survey vs simulation")
ax.legend(title="Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

The scatter above is a calibration check, not the main decision. If the simulation scores are all near `1.0` while the survey scores are around neutral or negative, the benchmark may still rank policies correctly, but its absolute score scale is not calibrated to human perception.

## Paired Difference Analysis

This is the core mixed analysis. For the survey, each difference is computed within participant. For the simulation, each difference is computed within episode slot. Then we compare the direction of those differences.

We use small practical deadbands so tiny numerical differences are treated as ties:

- Survey normalized delta threshold: `0.03`, which is about `0.18` points on the original `-3..+3` scale.
- Simulation delta threshold: `0.02` on the `0..1` benchmark scale.

These thresholds are not magic. They are there to avoid overinterpreting tiny differences.

In [ ]:
SURVEY_DELTA_THRESHOLD = 0.03
SIMULATION_DELTA_THRESHOLD = 0.02


def percentile_bootstrap_ci(values, statistic="mean", n_resamples=5000, confidence=0.95, seed=RANDOM_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan
    if len(values) == 1:
        return float(values[0]), float(values[0])

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    samples = values[sample_indices]
    if statistic == "median":
        bootstrap_statistics = np.median(samples, axis=1)
    else:
        bootstrap_statistics = np.mean(samples, axis=1)

    alpha = 1 - confidence
    return tuple(np.quantile(bootstrap_statistics, [alpha / 2, 1 - alpha / 2]))


def classify_direction(value, threshold):
    if pd.isna(value):
        return "missing"
    if value > threshold:
        return "left_higher"
    if value < -threshold:
        return "right_higher"
    return "tie"


def compare_directions(survey_direction, simulation_direction):
    if "missing" in {survey_direction, simulation_direction}:
        return "missing"
    if survey_direction == simulation_direction:
        return "agree"
    if "tie" in {survey_direction, simulation_direction}:
        return "one_tie"
    return "disagree"


def advantage_from_deltas(deltas, threshold):
    deltas = np.asarray(deltas, dtype=float)
    deltas = deltas[~np.isnan(deltas)]
    if len(deltas) == 0:
        return np.nan, 0, 0, 0
    wins = int((deltas > threshold).sum())
    ties = int((np.abs(deltas) <= threshold).sum())
    losses = int((deltas < -threshold).sum())
    relative_advantage = (wins + 0.5 * ties) / len(deltas)
    return relative_advantage, wins, ties, losses

In [ ]:
comparison_rows = []

for left_condition, right_condition, comparison_name, meaning in COMPARISONS:
    for axis in AXES_WITH_GLOBAL:
        left_survey = (
            survey_analysis_long[(survey_analysis_long["condition"] == left_condition) & (survey_analysis_long["axis"] == axis)]
            .set_index("participant_id")["score_norm"]
            .rename("left")
        )
        right_survey = (
            survey_analysis_long[(survey_analysis_long["condition"] == right_condition) & (survey_analysis_long["axis"] == axis)]
            .set_index("participant_id")["score_norm"]
            .rename("right")
        )
        survey_pair = pd.concat([left_survey, right_survey], axis=1, join="inner").dropna()
        survey_deltas = survey_pair["left"] - survey_pair["right"]
        survey_ci_low, survey_ci_high = percentile_bootstrap_ci(survey_deltas, statistic="mean")
        survey_advantage, survey_wins, survey_ties, survey_losses = advantage_from_deltas(survey_deltas, SURVEY_DELTA_THRESHOLD)

        left_sim = (
            simulation_analysis_long[(simulation_analysis_long["condition"] == left_condition) & (simulation_analysis_long["axis"] == axis)]
            .set_index("episode_slot")["sim_score"]
            .rename("left")
        )
        right_sim = (
            simulation_analysis_long[(simulation_analysis_long["condition"] == right_condition) & (simulation_analysis_long["axis"] == axis)]
            .set_index("episode_slot")["sim_score"]
            .rename("right")
        )
        simulation_pair = pd.concat([left_sim, right_sim], axis=1, join="inner").dropna()
        simulation_deltas = simulation_pair["left"] - simulation_pair["right"]
        simulation_advantage, simulation_wins, simulation_ties, simulation_losses = advantage_from_deltas(simulation_deltas, SIMULATION_DELTA_THRESHOLD)

        survey_mean_delta = survey_deltas.mean()
        simulation_mean_delta = simulation_deltas.mean()
        survey_direction = classify_direction(survey_mean_delta, SURVEY_DELTA_THRESHOLD)
        simulation_direction = classify_direction(simulation_mean_delta, SIMULATION_DELTA_THRESHOLD)

        comparison_rows.append(
            {
                "comparison": comparison_name,
                "meaning": meaning,
                "axis": axis,
                "left": left_condition,
                "right": right_condition,
                "survey_n_participants": len(survey_deltas),
                "survey_mean_delta_norm": survey_mean_delta,
                "survey_mean_delta_ci_low": survey_ci_low,
                "survey_mean_delta_ci_high": survey_ci_high,
                "survey_direction": survey_direction,
                "survey_ci_excludes_zero": (survey_ci_low > 0) or (survey_ci_high < 0),
                "survey_relative_advantage": survey_advantage,
                "survey_wins": survey_wins,
                "survey_ties": survey_ties,
                "survey_losses": survey_losses,
                "simulation_n_episode_pairs": len(simulation_deltas),
                "simulation_mean_delta": simulation_mean_delta,
                "simulation_direction": simulation_direction,
                "simulation_relative_advantage": simulation_advantage,
                "simulation_wins": simulation_wins,
                "simulation_ties": simulation_ties,
                "simulation_losses": simulation_losses,
                "direction_agreement": compare_directions(survey_direction, simulation_direction),
            }
        )

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.sort_values(["axis", "comparison"]))

In [ ]:
global_comparisons = comparison_results[comparison_results["axis"] == "global"].copy()
positions = np.arange(len(global_comparisons))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(positions - 0.18, global_comparisons["survey_mean_delta_norm"], height=0.35, label="Survey delta")
ax.barh(positions + 0.18, global_comparisons["simulation_mean_delta"], height=0.35, label="Simulation delta")
ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(positions)
ax.set_yticklabels(global_comparisons["comparison"])
ax.set_xlabel("Left minus right difference")
ax.set_title("Global paired differences: survey vs simulation")
ax.legend()
plt.tight_layout()
plt.show()

display(global_comparisons[["comparison", "survey_mean_delta_norm", "survey_mean_delta_ci_low", "survey_mean_delta_ci_high", "simulation_mean_delta", "direction_agreement"]])

## Ranking Alignment

This section checks whether the four conditions are ordered similarly by the survey and the simulation. With only four conditions per axis, the rank correlation is descriptive. It is useful as a compact sanity check, not as a final proof.

In [ ]:
def rank_correlation(left_values, right_values):
    data = pd.DataFrame({"left": left_values, "right": right_values}).dropna()
    if len(data) < 2:
        return np.nan
    left_ranks = data["left"].rank()
    right_ranks = data["right"].rank()
    if left_ranks.nunique() < 2 or right_ranks.nunique() < 2:
        return np.nan
    return left_ranks.corr(right_ranks)


rank_rows = []
for axis in AXES_WITH_GLOBAL:
    axis_frame = joint_summary[joint_summary["axis"] == axis].dropna(subset=["survey_mean_0_1", "sim_mean_0_1"]).copy()
    rank_rows.append(
        {
            "axis": axis,
            "n_conditions": len(axis_frame),
            "rank_correlation": rank_correlation(axis_frame["survey_mean_0_1"], axis_frame["sim_mean_0_1"]),
            "survey_order_high_to_low": " > ".join(axis_frame.sort_values("survey_mean_0_1", ascending=False)["condition"].str.upper()),
            "simulation_order_high_to_low": " > ".join(axis_frame.sort_values("sim_mean_0_1", ascending=False)["condition"].str.upper()),
        }
    )

rank_alignment = pd.DataFrame(rank_rows)
overall_rank_correlation = rank_correlation(plot_data["survey_mean_0_1"], plot_data["sim_mean_0_1"])

display(rank_alignment)
print(f"Overall descriptive rank correlation across available condition-axis points: {overall_rank_correlation:.3f}")

## Did We Do It Right Or Wrong?

This should be read as a benchmark validation check, not as a moral yes/no. The benchmark is working for this dataset when it predicts the same relative outcomes as the survey. It is not working, or is incomplete, where the survey and simulation disagree in direction or where the simulation has too little evidence.

In [ ]:
agreement_summary = (
    comparison_results.groupby(["axis", "direction_agreement"])
    .size()
    .unstack(fill_value=0)
    .reindex(AXES_WITH_GLOBAL)
)

decisive_results = comparison_results[comparison_results["direction_agreement"] != "missing"].copy()
agreement_rate = (decisive_results["direction_agreement"] == "agree").mean()

mismatches = comparison_results[comparison_results["direction_agreement"].isin(["disagree", "one_tie", "missing"])].copy()

display(agreement_summary)
print(f"Agreement rate across comparison rows: {agreement_rate:.1%}")

display(
    mismatches[
        [
            "comparison",
            "axis",
            "survey_mean_delta_norm",
            "survey_mean_delta_ci_low",
            "survey_mean_delta_ci_high",
            "survey_direction",
            "simulation_mean_delta",
            "simulation_direction",
            "direction_agreement",
        ]
    ].sort_values(["axis", "comparison"])
)


## Axis-wise calibration: offset + scale

We calibrate the simulation axis scores onto the human survey scale using
only the tuning subset. The calibration is monotonic if beta > 0:

     calibrated_sim = alpha + beta * raw_sim

 This should be interpreted as a temporary scale calibration, not as a
 re-learning of the internal metric structure.


In [ ]:
CALIBRATION_TUNING_N = 50
CALIBRATION_RANDOM_SEED = RANDOM_SEED

# Calibrate only the four real axes, not "global".
CALIBRATION_AXES = AXES.copy()

rng = np.random.default_rng(CALIBRATION_RANDOM_SEED)

participant_ids = np.array(sorted(survey_analysis_long["participant_id"].dropna().unique()))
rng.shuffle(participant_ids)

tuning_participants = set(participant_ids[:CALIBRATION_TUNING_N])
validation_participants = set(participant_ids[CALIBRATION_TUNING_N:])

print(f"Tuning participants: {len(tuning_participants)}")
print(f"Validation participants: {len(validation_participants)}")

survey_tuning_long = survey_analysis_long[
    survey_analysis_long["participant_id"].isin(tuning_participants)
].copy()

survey_validation_long = survey_analysis_long[
    survey_analysis_long["participant_id"].isin(validation_participants)
].copy()

# %%
def summarize_survey_subset(survey_subset_long, value_column="score_norm"):
    """Mean human score per condition x axis for one participant subset."""
    return (
        survey_subset_long[survey_subset_long["axis"].isin(CALIBRATION_AXES)]
        .groupby(["condition", "condition_label", "policy", "pov", "axis"], dropna=False)
        .agg(
            survey_mean_0_1=(value_column, "mean"),
            survey_std_0_1=(value_column, "std"),
            survey_n=(value_column, "count"),
        )
        .reset_index()
    )


def summarize_simulation_for_calibration(simulation_long, value_column="sim_score"):
    """Mean simulation score per condition x axis."""
    return (
        simulation_long[simulation_long["axis"].isin(CALIBRATION_AXES)]
        .groupby(["condition", "condition_label", "policy", "pov", "axis"], dropna=False)
        .agg(
            sim_mean_0_1=(value_column, "mean"),
            sim_std_0_1=(value_column, "std"),
            sim_n=(value_column, "count"),
        )
        .reset_index()
    )


survey_tuning_summary = summarize_survey_subset(survey_tuning_long)
survey_validation_summary = summarize_survey_subset(survey_validation_long)
simulation_calibration_summary = summarize_simulation_for_calibration(simulation_analysis_long)

calibration_tuning_table = survey_tuning_summary.merge(
    simulation_calibration_summary,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="inner",
)

calibration_validation_table = survey_validation_summary.merge(
    simulation_calibration_summary,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="inner",
)

display(calibration_tuning_table.sort_values(["axis", "condition"]))

In [ ]:
# %%
def fit_affine_calibration(x, y, beta_bounds=(0.0, 3.0)):
    """
    Fit y ~= alpha + beta * x.

    x: raw simulation score, expected in 0..1
    y: human survey mean, expected in 0..1

    beta is clipped to beta_bounds to avoid non-monotonic or absurd calibration.
    """
    data = pd.DataFrame({"x": x, "y": y}).dropna()

    if len(data) < 2 or data["x"].nunique() < 2:
        # Fallback: offset-only calibration.
        beta = 1.0
        alpha = float(data["y"].mean() - data["x"].mean()) if len(data) else 0.0
        fit_mode = "offset_only_fallback"
    else:
        beta, alpha = np.polyfit(data["x"], data["y"], deg=1)
        beta = float(np.clip(beta, beta_bounds[0], beta_bounds[1]))

        # Recompute alpha after beta clipping.
        alpha = float(data["y"].mean() - beta * data["x"].mean())
        fit_mode = "affine"

    return alpha, beta, fit_mode


def apply_affine_calibration(x, alpha, beta, clip=True):
    calibrated = alpha + beta * x
    if clip:
        calibrated = np.clip(calibrated, 0.0, 1.0)
    return calibrated


calibration_param_rows = []

for axis in CALIBRATION_AXES:
    axis_train = calibration_tuning_table[calibration_tuning_table["axis"] == axis].copy()

    alpha, beta, fit_mode = fit_affine_calibration(
        x=axis_train["sim_mean_0_1"],
        y=axis_train["survey_mean_0_1"],
        beta_bounds=(0.0, 3.0),
    )

    axis_train["sim_calibrated_0_1"] = apply_affine_calibration(
        axis_train["sim_mean_0_1"], alpha, beta
    )

    raw_mae_train = np.mean(np.abs(axis_train["sim_mean_0_1"] - axis_train["survey_mean_0_1"]))
    calibrated_mae_train = np.mean(np.abs(axis_train["sim_calibrated_0_1"] - axis_train["survey_mean_0_1"]))

    calibration_param_rows.append(
        {
            "axis": axis,
            "alpha": alpha,
            "beta": beta,
            "fit_mode": fit_mode,
            "n_conditions_used": len(axis_train),
            "raw_mae_train_0_1": raw_mae_train,
            "calibrated_mae_train_0_1": calibrated_mae_train,
            "raw_mae_train_0_100": raw_mae_train * 100,
            "calibrated_mae_train_0_100": calibrated_mae_train * 100,
        }
    )

calibration_params = pd.DataFrame(calibration_param_rows)

display(calibration_params)

In [ ]:
# %%
simulation_analysis_long_calibrated = simulation_analysis_long.copy()

simulation_analysis_long_calibrated["sim_score_calibrated"] = simulation_analysis_long_calibrated["sim_score"]

for _, row in calibration_params.iterrows():
    axis = row["axis"]
    alpha = row["alpha"]
    beta = row["beta"]

    mask = simulation_analysis_long_calibrated["axis"] == axis
    simulation_analysis_long_calibrated.loc[mask, "sim_score_calibrated"] = apply_affine_calibration(
        simulation_analysis_long_calibrated.loc[mask, "sim_score"],
        alpha,
        beta,
    )

# Optional: recompute calibrated global as the mean of calibrated axes.
# This is only useful if you still keep the global rows around for plots.
calibrated_axis_rows = simulation_analysis_long_calibrated[
    simulation_analysis_long_calibrated["axis"].isin(CALIBRATION_AXES)
].copy()

calibrated_global = (
    calibrated_axis_rows
    .groupby(["condition", "condition_label", "policy", "pov", "episode_slot", "episode_type"], dropna=False)
    .agg(
        sim_score=("sim_score", "mean"),
        sim_score_calibrated=("sim_score_calibrated", "mean"),
    )
    .reset_index()
)
calibrated_global["axis"] = "global"

simulation_analysis_long_calibrated = pd.concat(
    [
        simulation_analysis_long_calibrated[
            simulation_analysis_long_calibrated["axis"] != "global"
        ],
        calibrated_global,
    ],
    ignore_index=True,
    sort=False,
)

display(simulation_analysis_long_calibrated.head())

In [ ]:
# %%
simulation_summary_calibrated = (
    simulation_analysis_long_calibrated
    .groupby(["condition", "condition_label", "policy", "pov", "axis"], dropna=False)
    .agg(
        sim_mean_0_1=("sim_score", "mean"),
        sim_calibrated_mean_0_1=("sim_score_calibrated", "mean"),
        sim_median_0_1=("sim_score", "median"),
        sim_calibrated_median_0_1=("sim_score_calibrated", "median"),
        sim_std_0_1=("sim_score", "std"),
        sim_calibrated_std_0_1=("sim_score_calibrated", "std"),
        n_episode_values=("sim_score", "count"),
        n_episode_slots=("episode_slot", "nunique"),
    )
    .reset_index()
)

joint_tuning_calibrated = survey_tuning_summary.merge(
    simulation_summary_calibrated,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="inner",
)

joint_validation_calibrated = survey_validation_summary.merge(
    simulation_summary_calibrated,
    on=["condition", "condition_label", "policy", "pov", "axis"],
    how="inner",
)

for df in [joint_tuning_calibrated, joint_validation_calibrated]:
    df["raw_gap_sim_minus_survey"] = df["sim_mean_0_1"] - df["survey_mean_0_1"]
    df["calibrated_gap_sim_minus_survey"] = df["sim_calibrated_mean_0_1"] - df["survey_mean_0_1"]
    df["raw_abs_error"] = df["raw_gap_sim_minus_survey"].abs()
    df["calibrated_abs_error"] = df["calibrated_gap_sim_minus_survey"].abs()

display(joint_validation_calibrated.sort_values(["axis", "condition"]))

In [ ]:
# %%
def rank_correlation(left_values, right_values):
    data = pd.DataFrame({"left": left_values, "right": right_values}).dropna()
    if len(data) < 2:
        return np.nan

    left_ranks = data["left"].rank()
    right_ranks = data["right"].rank()

    if left_ranks.nunique() < 2 or right_ranks.nunique() < 2:
        return np.nan

    return left_ranks.corr(right_ranks)


def calibration_evaluation_table(joint_table, split_name):
    rows = []

    for axis in CALIBRATION_AXES:
        axis_df = joint_table[joint_table["axis"] == axis].copy()

        rows.append(
            {
                "split": split_name,
                "axis": axis,
                "n_condition_points": len(axis_df),
                "raw_mae_0_1": axis_df["raw_abs_error"].mean(),
                "calibrated_mae_0_1": axis_df["calibrated_abs_error"].mean(),
                "raw_mae_0_100": axis_df["raw_abs_error"].mean() * 100,
                "calibrated_mae_0_100": axis_df["calibrated_abs_error"].mean() * 100,
                "raw_bias_0_100": axis_df["raw_gap_sim_minus_survey"].mean() * 100,
                "calibrated_bias_0_100": axis_df["calibrated_gap_sim_minus_survey"].mean() * 100,
                "raw_rank_corr": rank_correlation(axis_df["survey_mean_0_1"], axis_df["sim_mean_0_1"]),
                "calibrated_rank_corr": rank_correlation(axis_df["survey_mean_0_1"], axis_df["sim_calibrated_mean_0_1"]),
            }
        )

    return pd.DataFrame(rows)


calibration_evaluation = pd.concat(
    [
        calibration_evaluation_table(joint_tuning_calibrated, "tuning"),
        calibration_evaluation_table(joint_validation_calibrated, "validation"),
    ],
    ignore_index=True,
)

display(calibration_evaluation)

In [ ]:
# %%
plot_cal = joint_validation_calibrated.copy()

fig, ax = plt.subplots(figsize=(7, 7))

for axis_name, axis_frame in plot_cal.groupby("axis"):
    ax.scatter(
        axis_frame["survey_mean_0_1"],
        axis_frame["sim_mean_0_1"],
        label=f"{axis_name} raw",
        marker="o",
        s=70,
        alpha=0.7,
    )
    ax.scatter(
        axis_frame["survey_mean_0_1"],
        axis_frame["sim_calibrated_mean_0_1"],
        label=f"{axis_name} calibrated",
        marker="x",
        s=90,
    )

ax.axline((0, 0), slope=1, color="black", linewidth=1, linestyle="--")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Survey validation mean, normalized 0..1")
ax.set_ylabel("Simulation score, raw/calibrated 0..1")
ax.set_title("Validation: raw vs calibrated simulation scores")
ax.legend(title="Axis", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# %%
CALIBRATED_SIMULATION_DELTA_THRESHOLD = SIMULATION_DELTA_THRESHOLD

calibrated_comparison_rows = []

for left_condition, right_condition, comparison_name, meaning in COMPARISONS:
    for axis in CALIBRATION_AXES:
        left_survey = (
            survey_validation_long[
                (survey_validation_long["condition"] == left_condition)
                & (survey_validation_long["axis"] == axis)
            ]
            .set_index("participant_id")["score_norm"]
            .rename("left")
        )

        right_survey = (
            survey_validation_long[
                (survey_validation_long["condition"] == right_condition)
                & (survey_validation_long["axis"] == axis)
            ]
            .set_index("participant_id")["score_norm"]
            .rename("right")
        )

        survey_pair = pd.concat([left_survey, right_survey], axis=1, join="inner").dropna()
        survey_deltas = survey_pair["left"] - survey_pair["right"]

        left_sim = (
            simulation_analysis_long_calibrated[
                (simulation_analysis_long_calibrated["condition"] == left_condition)
                & (simulation_analysis_long_calibrated["axis"] == axis)
            ]
            .set_index("episode_slot")["sim_score_calibrated"]
            .rename("left")
        )

        right_sim = (
            simulation_analysis_long_calibrated[
                (simulation_analysis_long_calibrated["condition"] == right_condition)
                & (simulation_analysis_long_calibrated["axis"] == axis)
            ]
            .set_index("episode_slot")["sim_score_calibrated"]
            .rename("right")
        )

        simulation_pair = pd.concat([left_sim, right_sim], axis=1, join="inner").dropna()
        simulation_deltas = simulation_pair["left"] - simulation_pair["right"]

        survey_mean_delta = survey_deltas.mean()
        simulation_mean_delta = simulation_deltas.mean()

        survey_direction = classify_direction(survey_mean_delta, SURVEY_DELTA_THRESHOLD)
        simulation_direction = classify_direction(simulation_mean_delta, CALIBRATED_SIMULATION_DELTA_THRESHOLD)

        calibrated_comparison_rows.append(
            {
                "comparison": comparison_name,
                "axis": axis,
                "left": left_condition,
                "right": right_condition,
                "survey_validation_n": len(survey_deltas),
                "survey_validation_mean_delta": survey_mean_delta,
                "simulation_calibrated_mean_delta": simulation_mean_delta,
                "survey_direction": survey_direction,
                "simulation_calibrated_direction": simulation_direction,
                "direction_agreement": compare_directions(survey_direction, simulation_direction),
            }
        )

calibrated_comparison_results = pd.DataFrame(calibrated_comparison_rows)

display(
    calibrated_comparison_results.sort_values(["axis", "comparison"])
)

## Held-out Calibration Figure

The calibration parameters above are learned only from `tuning_participants`. The table and radar below report the held-out validation/test participants before calibration and after applying the calibration, so the post-calibration scores are not training-set fit quality.


In [ ]:
VALIDATION_FIGURE_PATH = GENERATED_DIR / "heldout_calibration_comparison.png"
VALIDATION_TABLE_PATH = VALIDATION_FIGURE_PATH.with_suffix(".csv")
VALIDATION_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)

AXIS_DISPLAY_LABELS = {
    "competence_control": "Dexterity",
    "safety": "Safety",
    "awareness": "Social Awaren.",
    "positive_impression": "Impression",
}
POLICY_DISPLAY_LABELS = {"A": "Policy A", "B": "Policy B"}
STAGE_ORDER = ["Before calibration", "After calibration"]


def heldout_evaluation_frame(simulation_summary_frame, stage_label):
    """Join held-out human means to raw or calibrated simulation means."""
    frame = survey_validation_summary.merge(
        simulation_summary_frame,
        on=["condition", "condition_label", "policy", "pov", "axis"],
        how="inner",
    ).copy()
    frame["stage"] = stage_label
    frame["abs_error_0_100"] = (
        frame["sim_mean_0_1"] - frame["survey_mean_0_1"]
    ).abs() * 100
    return frame


calibrated_simulation_summary = summarize_simulation_for_calibration(
    simulation_analysis_long_calibrated,
    value_column="sim_score_calibrated",
)

heldout_raw_evaluation = heldout_evaluation_frame(
    simulation_calibration_summary,
    "Before calibration",
)
heldout_calibrated_evaluation = heldout_evaluation_frame(
    calibrated_simulation_summary,
    "After calibration",
)
heldout_evaluation = pd.concat(
    [heldout_raw_evaluation, heldout_calibrated_evaluation],
    ignore_index=True,
    sort=False,
)

heldout_policy_axis = (
    heldout_evaluation.groupby(["stage", "policy", "axis"], dropna=False)
    .agg(
        human_mean_0_1=("survey_mean_0_1", "mean"),
        auto_mean_0_1=("sim_mean_0_1", "mean"),
        mean_abs_error_0_100=("abs_error_0_100", "mean"),
        n_conditions=("condition", "nunique"),
    )
    .reset_index()
)


def score_lookup(stage, policy, axis, column):
    match = heldout_policy_axis[
        (heldout_policy_axis["stage"] == stage)
        & (heldout_policy_axis["policy"] == policy)
        & (heldout_policy_axis["axis"] == axis)
    ]
    if match.empty:
        return np.nan
    return float(match.iloc[0][column]) * 100


def alignment_lookup(stage, axis=None):
    frame = heldout_policy_axis[heldout_policy_axis["stage"] == stage]
    if axis is not None:
        frame = frame[frame["axis"] == axis]
    if frame.empty:
        return np.nan
    return float(100 - frame["mean_abs_error_0_100"].mean())


comparison_rows = []
for stage in STAGE_ORDER:
    for axis in CALIBRATION_AXES:
        comparison_rows.append(
            {
                "Stage": stage,
                "Metric": AXIS_DISPLAY_LABELS[axis],
                "Policy A Human": score_lookup(stage, "A", axis, "human_mean_0_1"),
                "Policy A Auto": score_lookup(stage, "A", axis, "auto_mean_0_1"),
                "Policy B Human": score_lookup(stage, "B", axis, "human_mean_0_1"),
                "Policy B Auto": score_lookup(stage, "B", axis, "auto_mean_0_1"),
                "Alignment Score": alignment_lookup(stage, axis),
            }
        )

    comparison_rows.append(
        {
            "Stage": stage,
            "Metric": "AsimovBM",
            "Policy A Human": heldout_policy_axis[
                (heldout_policy_axis["stage"] == stage)
                & (heldout_policy_axis["policy"] == "A")
            ]["human_mean_0_1"].mean()
            * 100,
            "Policy A Auto": heldout_policy_axis[
                (heldout_policy_axis["stage"] == stage)
                & (heldout_policy_axis["policy"] == "A")
            ]["auto_mean_0_1"].mean()
            * 100,
            "Policy B Human": heldout_policy_axis[
                (heldout_policy_axis["stage"] == stage)
                & (heldout_policy_axis["policy"] == "B")
            ]["human_mean_0_1"].mean()
            * 100,
            "Policy B Auto": heldout_policy_axis[
                (heldout_policy_axis["stage"] == stage)
                & (heldout_policy_axis["policy"] == "B")
            ]["auto_mean_0_1"].mean()
            * 100,
            "Alignment Score": alignment_lookup(stage),
        }
    )

validation_comparison_table = pd.DataFrame(comparison_rows)
validation_comparison_table_rounded = validation_comparison_table.copy()
score_columns = [column for column in validation_comparison_table.columns if column not in {"Stage", "Metric"}]
validation_comparison_table_rounded[score_columns] = validation_comparison_table_rounded[
    score_columns
].round(1)
validation_comparison_table_rounded.to_csv(VALIDATION_TABLE_PATH, index=False)

validation_human_groups = (
    survey_validation_long[
        survey_validation_long["axis"].isin(CALIBRATION_AXES)
        & survey_validation_long["robotics_familiarity"].notna()
    ]
    .assign(robotics_familiarity=lambda frame: frame["robotics_familiarity"].astype(bool))
    .groupby(["robotics_familiarity", "axis"], dropna=False)["score_norm"]
    .mean()
    .unstack("axis")
    .reindex(columns=CALIBRATION_AXES)
    .mul(100)
)

auto_axis_by_stage = (
    heldout_evaluation.groupby(["stage", "axis"], dropna=False)["sim_mean_0_1"]
    .mean()
    .unstack("axis")
    .reindex(index=STAGE_ORDER, columns=CALIBRATION_AXES)
    .mul(100)
)

angles = np.linspace(0, 2 * np.pi, len(CALIBRATION_AXES), endpoint=False).tolist()
angles += angles[:1]
radar_labels = [AXIS_DISPLAY_LABELS[axis].replace(" ", "\n") for axis in CALIBRATION_AXES]


def closed_values(values):
    values = [float(value) for value in values]
    return values + values[:1]


def plot_validation_radar(ax, stage):
    series = [
        (
            "Auto evaluator",
            auto_axis_by_stage.loc[stage],
            "#1f77b4",
        ),
        (
            "Human test\n(no daily contact)",
            validation_human_groups.loc[False],
            "#f2b134",
        ),
        (
            "Human test\n(daily robotics)",
            validation_human_groups.loc[True],
            "#2eb67d",
        ),
    ]
    for label, values, color in series:
        radar_values = closed_values(values.reindex(CALIBRATION_AXES).to_numpy())
        ax.plot(angles, radar_values, color=color, linewidth=2, marker="o", label=label)
        ax.fill(angles, radar_values, color=color, alpha=0.08)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_labels, fontsize=9)
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(["25", "50", "75", "100"], fontsize=8)
    ax.set_title(stage, fontsize=12, pad=14)
    ax.grid(True, linewidth=0.8, alpha=0.45)


fig = plt.figure(figsize=(14, 10), constrained_layout=True)
grid = fig.add_gridspec(nrows=2, ncols=2, height_ratios=[1.1, 1.35])

ax_table = fig.add_subplot(grid[0, :])
ax_table.axis("off")
ax_table.set_title(
    "Held-out validation/test comparison of automated and human H evaluation",
    fontsize=14,
    fontweight="bold",
    pad=12,
)

figure_table = validation_comparison_table_rounded.copy()
figure_table["Stage"] = figure_table["Stage"].str.replace(" calibration", "", regex=False)
cell_text = figure_table.astype(str).values.tolist()
column_labels = figure_table.columns.tolist()
matplotlib_table = ax_table.table(
    cellText=cell_text,
    colLabels=column_labels,
    loc="center",
    cellLoc="center",
    colLoc="center",
)
matplotlib_table.auto_set_font_size(False)
matplotlib_table.set_fontsize(8.5)
matplotlib_table.scale(1, 1.35)
for (row_index, column_index), cell in matplotlib_table.get_celld().items():
    cell.set_edgecolor("#444444")
    cell.set_linewidth(0.6)
    if row_index == 0:
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f1f3f5")
    elif figure_table.iloc[row_index - 1]["Metric"] == "AsimovBM":
        cell.set_text_props(weight="bold")
        cell.set_facecolor("#f8f9fa")

before_axis = fig.add_subplot(grid[1, 0], polar=True)
after_axis = fig.add_subplot(grid[1, 1], polar=True)
plot_validation_radar(before_axis, "Before calibration")
plot_validation_radar(after_axis, "After calibration")
after_axis.legend(loc="lower center", bbox_to_anchor=(0.5, -0.28), ncol=3, fontsize=8)

fig.savefig(VALIDATION_FIGURE_PATH, dpi=220, bbox_inches="tight")
plt.show()

print(f"Saved held-out validation figure: {VALIDATION_FIGURE_PATH}")
print(f"Saved held-out validation table: {VALIDATION_TABLE_PATH}")
display(validation_comparison_table_rounded)


![Held-out calibration table and radar](../generated/heldout_calibration_comparison.png)


In [ ]:
# Table 1 data for the paper.
# Source: paper/generated/heldout_calibration_comparison.csv
PAPER_TABLE_1_PATH = GENERATED_DIR / "heldout_calibration_comparison.csv"

paper_table_1 = pd.read_csv(PAPER_TABLE_1_PATH)
paper_table_1_score_columns = [
    "Policy A Human",
    "Policy A Auto",
    "Policy B Human",
    "Policy B Auto",
    "Alignment Score",
]
paper_table_1[paper_table_1_score_columns] = paper_table_1[
    paper_table_1_score_columns
].apply(pd.to_numeric).round(1)

display(paper_table_1)
print(f"Paper Table 1 source: {PAPER_TABLE_1_PATH}")
print(paper_table_1.to_csv(index=False))


## Paper-ready Spiderplot With 95% Confidence Intervals

This single spiderplot is intended for the paper draft. It uses the held-out validation/test set after calibration, zooms the radial scale to `25..50`, and shows 95% bootstrap confidence intervals as dashed inner/outer envelopes around each mean line.


In [ ]:
PAPER_SPIDERPLOT_PATH = GENERATED_DIR / "heldout_calibration_spiderplot_ci.png"
PAPER_SPIDERPLOT_CI_PATH = PAPER_SPIDERPLOT_PATH.with_suffix(".csv")
PAPER_SPIDERPLOT_PATH.parent.mkdir(parents=True, exist_ok=True)

PAPER_SPIDERPLOT_R_MIN = 25
PAPER_SPIDERPLOT_R_MAX = 50
PAPER_SPIDERPLOT_BOOTSTRAPS = 10000
PAPER_SPIDERPLOT_SEED = RANDOM_SEED + 17

PAPER_AXIS_LABELS = {
    "competence_control": "Dexterity",
    "safety": "Safety",
    "awareness": "Social\nAware.",
    "positive_impression": "Impression",
}


def bootstrap_mean_ci(values, *, n_resamples=PAPER_SPIDERPLOT_BOOTSTRAPS, seed=PAPER_SPIDERPLOT_SEED):
    """Return mean and percentile 95% CI for one vector of 0..100 scores."""
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    if len(values) == 1:
        value = float(values[0])
        return value, value, value, 1

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    bootstrap_means = values[sample_indices].mean(axis=1)
    ci_low, ci_high = np.quantile(bootstrap_means, [0.025, 0.975])
    return float(values.mean()), float(ci_low), float(ci_high), int(len(values))


def participant_axis_values(survey_subset):
    """Participant-level axis means keep repeated condition ratings paired."""
    return (
        survey_subset[survey_subset["axis"].isin(CALIBRATION_AXES)]
        .groupby(["participant_id", "axis"], dropna=False)
        .agg(score_0_100=("score_norm", lambda values: values.mean() * 100))
        .reset_index()
    )


known_validation_humans = survey_validation_long[
    survey_validation_long["robotics_familiarity"].notna()
].copy()
known_validation_humans["robotics_familiarity_bool"] = known_validation_humans[
    "robotics_familiarity"
].astype(bool)

paper_series_specs = [
    {
        "series": "Auto evaluator",
        "kind": "auto",
        "color": "#1f77b4",
        "legend_label": "Auto",
    },
    {
        "series": "Human test (no daily contact)",
        "kind": "human",
        "robotics_familiarity": False,
        "color": "#f2b134",
        "legend_label": "Human (no daily)",
    },
    {
        "series": "Human test (daily robotics)",
        "kind": "human",
        "robotics_familiarity": True,
        "color": "#2eb67d",
        "legend_label": "Human (daily robotics)",
    },
]

paper_ci_rows = []
for spec_index, spec in enumerate(paper_series_specs):
    if spec["kind"] == "auto":
        source_frame = simulation_analysis_long_calibrated[
            simulation_analysis_long_calibrated["axis"].isin(CALIBRATION_AXES)
        ].copy()
        for axis in CALIBRATION_AXES:
            values = (
                source_frame[source_frame["axis"] == axis]["sim_score_calibrated"]
                .dropna()
                .to_numpy()
                * 100
            )
            mean, ci_low, ci_high, n = bootstrap_mean_ci(values, seed=PAPER_SPIDERPLOT_SEED + spec_index)
            paper_ci_rows.append(
                {
                    "series": spec["series"],
                    "axis": axis,
                    "mean_0_100": mean,
                    "ci_low_0_100": ci_low,
                    "ci_high_0_100": ci_high,
                    "n": n,
                    "bootstrap_unit": "condition_episode",
                }
            )
    else:
        group_frame = known_validation_humans[
            known_validation_humans["robotics_familiarity_bool"] == spec["robotics_familiarity"]
        ]
        participant_values = participant_axis_values(group_frame)
        for axis in CALIBRATION_AXES:
            values = participant_values[participant_values["axis"] == axis]["score_0_100"].to_numpy()
            mean, ci_low, ci_high, n = bootstrap_mean_ci(values, seed=PAPER_SPIDERPLOT_SEED + spec_index)
            paper_ci_rows.append(
                {
                    "series": spec["series"],
                    "axis": axis,
                    "mean_0_100": mean,
                    "ci_low_0_100": ci_low,
                    "ci_high_0_100": ci_high,
                    "n": n,
                    "bootstrap_unit": "participant",
                }
            )

paper_spiderplot_ci = pd.DataFrame(paper_ci_rows)
paper_spiderplot_ci_rounded = paper_spiderplot_ci.copy()
for column in ["mean_0_100", "ci_low_0_100", "ci_high_0_100"]:
    paper_spiderplot_ci_rounded[column] = paper_spiderplot_ci_rounded[column].round(1)
paper_spiderplot_ci_rounded.to_csv(PAPER_SPIDERPLOT_CI_PATH, index=False)

paper_angles = np.linspace(0, 2 * np.pi, len(CALIBRATION_AXES), endpoint=False).tolist()
paper_angles += paper_angles[:1]


def closed_axis_values(frame, column):
    values = [
        float(frame.loc[frame["axis"] == axis, column].iloc[0])
        for axis in CALIBRATION_AXES
    ]
    return values + values[:1]


fig, ax = plt.subplots(figsize=(7.1, 6.9), subplot_kw={"polar": True})

for spec in paper_series_specs:
    series_frame = paper_spiderplot_ci[paper_spiderplot_ci["series"] == spec["series"]]
    mean_values = closed_axis_values(series_frame, "mean_0_100")
    low_values = closed_axis_values(series_frame, "ci_low_0_100")
    high_values = closed_axis_values(series_frame, "ci_high_0_100")
    color = spec["color"]

    ax.fill_between(paper_angles, low_values, high_values, color=color, alpha=0.055, zorder=1)
    ax.plot(
        paper_angles,
        low_values,
        color=color,
        linewidth=1.65,
        linestyle=(0, (5, 4)),
        alpha=0.78,
        zorder=2,
    )
    ax.plot(
        paper_angles,
        high_values,
        color=color,
        linewidth=1.65,
        linestyle=(0, (5, 4)),
        alpha=0.78,
        zorder=2,
    )
    ax.plot(
        paper_angles,
        mean_values,
        color=color,
        linewidth=3.4,
        marker="o",
        markersize=8.5,
        label=spec.get("legend_label", spec["series"]),
        zorder=3,
    )

ax.set_ylim(PAPER_SPIDERPLOT_R_MIN, PAPER_SPIDERPLOT_R_MAX)
ax.set_yticks([25, 30, 35, 40, 45, 50])
ax.set_yticklabels(["25", "30", "35", "40", "45", "50"], fontsize=12)
ax.set_xticks(paper_angles[:-1])
ax.set_xticklabels([PAPER_AXIS_LABELS[axis] for axis in CALIBRATION_AXES], fontsize=15, fontweight="bold")
ax.tick_params(axis="x", pad=12)
ax.grid(True, linewidth=1.0, alpha=0.55)
ax.spines["polar"].set_linewidth(1.25)
ax.set_rlabel_position(25)
fig.subplots_adjust(left=0.17, right=0.83, top=0.86, bottom=0.29)
ax.set_title("Held-out test set after calibration", fontsize=14, fontweight="bold", pad=14)
legend = ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.5, -0.24),
    ncol=3,
    fontsize=10.5,
    frameon=True,
)
legend.get_frame().set_alpha(0.95)
legend.get_frame().set_linewidth(0.8)

fig.savefig(PAPER_SPIDERPLOT_PATH, dpi=450, bbox_inches="tight", pad_inches=0.18)
plt.show()

print(f"Saved paper-ready spiderplot: {PAPER_SPIDERPLOT_PATH}")
print(f"Saved spiderplot CI table: {PAPER_SPIDERPLOT_CI_PATH}")
display(paper_spiderplot_ci_rounded)


![Paper-ready held-out calibration spiderplot with 95% confidence intervals](../generated/heldout_calibration_spiderplot_ci.png)


## Paper-ready Policy Spiderplot

This cell creates a single paper figure with four polygons: human and calibrated automated scores for Policy A and Policy B. Human scores are aggregated across all held-out validation/test participants, without robotics-experience subgroups.


In [ ]:
from pathlib import Path
from matplotlib import font_manager

POLICY_SPIDERPLOT_PATH = GENERATED_DIR / "heldout_policy_spiderplot.png"
POLICY_SPIDERPLOT_TABLE_PATH = POLICY_SPIDERPLOT_PATH.with_suffix(".csv")
POLICY_SPIDERPLOT_PATH.parent.mkdir(parents=True, exist_ok=True)

POLICY_SPIDERPLOT_R_MIN = 10
POLICY_SPIDERPLOT_R_MAX = 60
policy_spiderplot_r_ticks = (
    [20, 30, 40, 50, 60]
    if POLICY_SPIDERPLOT_R_MIN == 20
    else [POLICY_SPIDERPLOT_R_MIN, 20, 30, 40, 50, 60]
)
POLICY_AXIS_LABELS = {
    "competence_control": "Dexterity",
    "safety": "Safety",
    "awareness": "Social\nAware.",
    "positive_impression": "Impression",
}
POLICY_SERIES = [
    {"policy": "A", "source": "Human", "label": "Policy A Human", "color": "#174A7E"},
    {"policy": "A", "source": "Auto", "label": "Policy A Auto", "color": "#4C9BD4"},
    {"policy": "B", "source": "Human", "label": "Policy B Human", "color": "#B45F06"},
    {"policy": "B", "source": "Auto", "label": "Policy B Auto", "color": "#E8A317"},
]

for font_path in Path("/usr/share/fonts/truetype/croscore").glob("Tinos*.ttf"):
    font_manager.fontManager.addfont(font_path)

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
if "Times New Roman" in available_fonts:
    POLICY_SPIDERPLOT_FONT = "Times New Roman"
elif "Tinos" in available_fonts:
    POLICY_SPIDERPLOT_FONT = "Tinos"
else:
    POLICY_SPIDERPLOT_FONT = "Liberation Serif"

policy_human_axis = (
    survey_validation_long[survey_validation_long["axis"].isin(CALIBRATION_AXES)]
    .groupby(["participant_id", "policy", "axis"], dropna=False)
    .agg(score_0_100=("score_norm", lambda values: values.mean() * 100))
    .reset_index()
    .groupby(["policy", "axis"], dropna=False)
    .agg(score_0_100=("score_0_100", "mean"), n_units=("participant_id", "count"))
    .reset_index()
)

policy_auto_axis = (
    simulation_analysis_long_calibrated[
        simulation_analysis_long_calibrated["axis"].isin(CALIBRATION_AXES)
    ]
    .groupby(["policy", "axis"], dropna=False)
    .agg(score_0_100=("sim_score_calibrated", lambda values: values.mean() * 100), n_units=("episode_slot", "count"))
    .reset_index()
)

policy_spiderplot_rows = []
for series in POLICY_SERIES:
    frame = policy_human_axis if series["source"] == "Human" else policy_auto_axis
    for axis in CALIBRATION_AXES:
        match = frame[(frame["policy"] == series["policy"]) & (frame["axis"] == axis)]
        policy_spiderplot_rows.append(
            {
                "series": series["label"],
                "policy": series["policy"],
                "source": series["source"],
                "axis": axis,
                "score_0_100": float(match.iloc[0]["score_0_100"]) if not match.empty else np.nan,
                "n_units": int(match.iloc[0]["n_units"]) if not match.empty else 0,
                "aggregation_unit": "participant" if series["source"] == "Human" else "episode_condition",
            }
        )

policy_spiderplot_table = pd.DataFrame(policy_spiderplot_rows)
policy_spiderplot_table_rounded = policy_spiderplot_table.copy()
policy_spiderplot_table_rounded["score_0_100"] = policy_spiderplot_table_rounded["score_0_100"].round(1)
policy_spiderplot_table_rounded.to_csv(POLICY_SPIDERPLOT_TABLE_PATH, index=False)

policy_angles = np.linspace(0, 2 * np.pi, len(CALIBRATION_AXES), endpoint=False).tolist()
policy_angles += policy_angles[:1]


def closed_policy_values(series_label):
    series_frame = policy_spiderplot_table[policy_spiderplot_table["series"] == series_label]
    values = [
        float(series_frame.loc[series_frame["axis"] == axis, "score_0_100"].iloc[0])
        for axis in CALIBRATION_AXES
    ]
    return values + values[:1]


with plt.rc_context(
    {
        "font.family": POLICY_SPIDERPLOT_FONT,
        "font.serif": ["Times New Roman", "Tinos", "Liberation Serif", "serif"],
        "mathtext.fontset": "dejavuserif",
    }
):
    fig, ax = plt.subplots(figsize=(6.9, 6.5), subplot_kw={"polar": True})

    for index, series in enumerate(POLICY_SERIES):
        values = closed_policy_values(series["label"])
        ax.plot(
            policy_angles,
            values,
            color=series["color"],
            linewidth=2.45,
            marker="o",
            markersize=5.8,
            label=series["label"],
            zorder=4 - index * 0.1,
        )

    ax.set_ylim(POLICY_SPIDERPLOT_R_MIN, POLICY_SPIDERPLOT_R_MAX)
    ax.set_yticks(policy_spiderplot_r_ticks)
    ax.set_yticklabels([f"{tick:g}" for tick in policy_spiderplot_r_ticks], fontsize=10.5)
    ax.set_xticks(policy_angles[:-1])
    ax.set_xticklabels([POLICY_AXIS_LABELS[axis] for axis in CALIBRATION_AXES], fontsize=12.5)
    ax.tick_params(axis="x", pad=9)
    ax.grid(True, linewidth=0.8, alpha=0.5)
    ax.spines["polar"].set_linewidth(1.0)
    ax.set_rlabel_position(25)
    legend = ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, -0.2),
        ncol=2,
        fontsize=10.5,
        frameon=True,
    )
    legend.get_frame().set_alpha(0.95)
    legend.get_frame().set_linewidth(0.7)
    fig.subplots_adjust(left=0.16, right=0.84, top=0.92, bottom=0.24)
    fig.savefig(POLICY_SPIDERPLOT_PATH, dpi=450, bbox_inches="tight", pad_inches=0.15)
    plt.show()

print(f"Saved policy spiderplot: {POLICY_SPIDERPLOT_PATH}")
print(f"Saved policy spiderplot table: {POLICY_SPIDERPLOT_TABLE_PATH}")
display(policy_spiderplot_table_rounded)


![Paper-ready held-out policy spiderplot](../generated/heldout_policy_spiderplot.png)


## Paper-ready Policy Spiderplot With Human Confidence Intervals

This copy keeps the four policy/source benchmark polygons unfilled and adds 95% bootstrap confidence bands only for the held-out human evaluations.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.legend_handler import HandlerTuple
from matplotlib.transforms import ScaledTranslation

POLICY_SPIDERPLOT_HUMAN_CI_PATH = GENERATED_DIR / "heldout_policy_spiderplot_human_ci.png"
POLICY_SPIDERPLOT_HUMAN_CI_SVG_PATH = POLICY_SPIDERPLOT_HUMAN_CI_PATH.with_suffix(".svg")
POLICY_SPIDERPLOT_HUMAN_CI_TABLE_PATH = POLICY_SPIDERPLOT_HUMAN_CI_PATH.with_suffix(".csv")
POLICY_HUMAN_CI_BOOTSTRAPS = 10000
POLICY_HUMAN_CI_SEED = RANDOM_SEED + 31
POLICY_HUMAN_CI_LEVEL = 95
POLICY_SPIDERPLOT_HUMAN_CI_R_MIN = 20  # Set anywhere from 0 to 20.
# Label nudges are in points. Positive x moves right; negative x moves left.
# Positive y moves up; negative y moves down.
POLICY_SPIDERPLOT_HUMAN_CI_DEXTERITY_X_OFFSET_POINTS = 1.0
POLICY_SPIDERPLOT_HUMAN_CI_SAFETY_Y_OFFSET_POINTS = -9.0
POLICY_SPIDERPLOT_HUMAN_CI_SOCIAL_AWARENESS_X_OFFSET_POINTS = -3.0
POLICY_SPIDERPLOT_HUMAN_CI_IMPRESSION_Y_OFFSET_POINTS = 0.0
POLICY_SPIDERPLOT_HUMAN_CI_LABEL_OFFSETS_POINTS = {
    "competence_control": (POLICY_SPIDERPLOT_HUMAN_CI_DEXTERITY_X_OFFSET_POINTS, 0.0),
    "safety": (0.0, POLICY_SPIDERPLOT_HUMAN_CI_SAFETY_Y_OFFSET_POINTS),
    "awareness": (POLICY_SPIDERPLOT_HUMAN_CI_SOCIAL_AWARENESS_X_OFFSET_POINTS, 0.0),
    "positive_impression": (0.0, POLICY_SPIDERPLOT_HUMAN_CI_IMPRESSION_Y_OFFSET_POINTS),
}
if not 0 <= POLICY_SPIDERPLOT_HUMAN_CI_R_MIN <= 20:
    raise ValueError("POLICY_SPIDERPLOT_HUMAN_CI_R_MIN must be between 0 and 20.")
policy_human_ci_r_ticks = (
    [20, 30, 40, 50, 60]
    if POLICY_SPIDERPLOT_HUMAN_CI_R_MIN == 20
    else [POLICY_SPIDERPLOT_HUMAN_CI_R_MIN, 20, 30, 40, 50, 60]
)


def policy_bootstrap_mean_ci(values, *, n_resamples=POLICY_HUMAN_CI_BOOTSTRAPS, seed=POLICY_HUMAN_CI_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    if len(values) == 1:
        value = float(values[0])
        return value, value, value, 1

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    bootstrap_means = values[sample_indices].mean(axis=1)
    ci_probability = POLICY_HUMAN_CI_LEVEL / 100
    tail_probability = (1 - ci_probability) / 2
    ci_low, ci_high = np.quantile(bootstrap_means, [tail_probability, 1 - tail_probability])
    return float(values.mean()), float(ci_low), float(ci_high), int(len(values))

policy_human_participant_values = (
    survey_validation_long[survey_validation_long["axis"].isin(CALIBRATION_AXES)]
    .groupby(["participant_id", "policy", "axis"], dropna=False)
    .agg(score_0_100=("score_norm", lambda values: values.mean() * 100))
    .reset_index()
)

policy_human_ci_rows = []
for policy_index, policy in enumerate(["A", "B"]):
    for axis_index, axis in enumerate(CALIBRATION_AXES):
        values = policy_human_participant_values[
            (policy_human_participant_values["policy"] == policy)
            & (policy_human_participant_values["axis"] == axis)
        ]["score_0_100"].to_numpy()
        mean, ci_low, ci_high, n = policy_bootstrap_mean_ci(
            values,
            n_resamples=POLICY_HUMAN_CI_BOOTSTRAPS,
            seed=POLICY_HUMAN_CI_SEED + policy_index * 10 + axis_index,
        )
        policy_human_ci_rows.append(
            {
                "series": f"Policy {policy} Human",
                "policy": policy,
                "source": "Human",
                "axis": axis,
                "mean_0_100": mean,
                "ci_low_0_100": ci_low,
                "ci_high_0_100": ci_high,
                "ci_level_percent": POLICY_HUMAN_CI_LEVEL,
                "n": n,
                "bootstrap_unit": "participant",
            }
        )

policy_human_ci = pd.DataFrame(policy_human_ci_rows)
policy_spiderplot_human_ci_table = policy_spiderplot_table.merge(
    policy_human_ci[
        ["series", "policy", "source", "axis", "ci_low_0_100", "ci_high_0_100", "ci_level_percent", "bootstrap_unit"]
    ],
    on=["series", "policy", "source", "axis"],
    how="left",
)
policy_spiderplot_human_ci_table["ci_low_0_100"] = policy_spiderplot_human_ci_table[
    "ci_low_0_100"
].where(policy_spiderplot_human_ci_table["source"] == "Human")
policy_spiderplot_human_ci_table["ci_high_0_100"] = policy_spiderplot_human_ci_table[
    "ci_high_0_100"
].where(policy_spiderplot_human_ci_table["source"] == "Human")
policy_spiderplot_human_ci_table["bootstrap_unit"] = policy_spiderplot_human_ci_table[
    "bootstrap_unit"
].where(policy_spiderplot_human_ci_table["source"] == "Human", "")
policy_spiderplot_human_ci_table["ci_level_percent"] = policy_spiderplot_human_ci_table[
    "ci_level_percent"
].where(policy_spiderplot_human_ci_table["source"] == "Human")

policy_spiderplot_human_ci_table_rounded = policy_spiderplot_human_ci_table.copy()
for column in ["score_0_100", "ci_low_0_100", "ci_high_0_100"]:
    policy_spiderplot_human_ci_table_rounded[column] = policy_spiderplot_human_ci_table_rounded[column].round(1)
policy_spiderplot_human_ci_table_rounded.to_csv(POLICY_SPIDERPLOT_HUMAN_CI_TABLE_PATH, index=False)


def closed_policy_human_ci_values(series_label, column):
    series_frame = policy_human_ci[policy_human_ci["series"] == series_label]
    values = [
        float(series_frame.loc[series_frame["axis"] == axis, column].iloc[0])
        for axis in CALIBRATION_AXES
    ]
    return values + values[:1]


with plt.rc_context(
    {
        "font.family": POLICY_SPIDERPLOT_FONT,
        "font.serif": ["Times New Roman", "Tinos", "Liberation Serif", "serif"],
        "mathtext.fontset": "dejavuserif",
    }
):
    fig, ax = plt.subplots(figsize=(6.9, 6.5), subplot_kw={"polar": True})
    ci_patch_handles = []

    for series in [spec for spec in POLICY_SERIES if spec["source"] == "Human"]:
        low_values = closed_policy_human_ci_values(series["label"], "ci_low_0_100")
        high_values = closed_policy_human_ci_values(series["label"], "ci_high_0_100")
        ax.fill_between(policy_angles, low_values, high_values, color=series["color"], alpha=0.105, zorder=1)
        ci_patch_handles.append(
            Patch(
                facecolor=series["color"],
                edgecolor="none",
                alpha=0.105,
                label=f"{series['label']} {POLICY_HUMAN_CI_LEVEL}% CI",
            )
        )

    for index, series in enumerate(POLICY_SERIES):
        values = closed_policy_values(series["label"])
        ax.plot(
            policy_angles,
            values,
            color=series["color"],
            linewidth=2.45,
            marker="o",
            markersize=5.8,
            label=series["label"],
            zorder=4 - index * 0.1,
        )

    ax.set_ylim(POLICY_SPIDERPLOT_HUMAN_CI_R_MIN, POLICY_SPIDERPLOT_R_MAX)
    ax.set_yticks(policy_human_ci_r_ticks)
    radial_tick_labels = ax.set_yticklabels([f"{tick:g}" for tick in policy_human_ci_r_ticks], fontsize=11.0)
    for label in radial_tick_labels:
        label.set_zorder(10)
        label.set_bbox({"facecolor": "white", "edgecolor": "none", "alpha": 0.72, "pad": 0.2})
    ax.set_xticks(policy_angles[:-1])
    axis_tick_labels = ax.set_xticklabels([POLICY_AXIS_LABELS[axis] for axis in CALIBRATION_AXES], fontsize=14)
    ax.tick_params(axis="x", pad=9)
    for axis, label in zip(CALIBRATION_AXES, axis_tick_labels):
        x_offset_points, y_offset_points = POLICY_SPIDERPLOT_HUMAN_CI_LABEL_OFFSETS_POINTS.get(axis, (0.0, 0.0))
        if x_offset_points or y_offset_points:
            label.set_transform(
                label.get_transform()
                + ScaledTranslation(x_offset_points / 72, y_offset_points / 72, fig.dpi_scale_trans)
            )
    ax.grid(True, linewidth=0.8, alpha=0.5)
    ax.spines["polar"].set_linewidth(1.0)
    ax.set_rlabel_position(25)
    line_handles, line_labels = ax.get_legend_handles_labels()
    line_handle_by_label = dict(zip(line_labels, line_handles))
    ci_handle_by_label = {handle.get_label(): handle for handle in ci_patch_handles}
    legend_handles = [
        (
            ci_handle_by_label[f"Policy A Human {POLICY_HUMAN_CI_LEVEL}% CI"],
            line_handle_by_label["Policy A Human"],
        ),
        line_handle_by_label["Policy A Auto"],
        (
            ci_handle_by_label[f"Policy B Human {POLICY_HUMAN_CI_LEVEL}% CI"],
            line_handle_by_label["Policy B Human"],
        ),
        line_handle_by_label["Policy B Auto"],
    ]
    legend_labels = [
        f"Policy A Human, {POLICY_HUMAN_CI_LEVEL}% CI",
        "Policy A Auto",
        f"Policy B Human, {POLICY_HUMAN_CI_LEVEL}% CI",
        "Policy B Auto",
    ]
    legend = ax.legend(
        handles=legend_handles,
        labels=legend_labels,
        handler_map={tuple: HandlerTuple(ndivide=1)},
        loc="lower center",
        bbox_to_anchor=(0.5, -0.35),
        ncol=2,
        fontsize=11,
        frameon=True,
    )
    legend.get_frame().set_alpha(0.95)
    legend.get_frame().set_linewidth(0.7)
    fig.subplots_adjust(left=0.16, right=0.84, top=0.92, bottom=0.36)
    fig.savefig(POLICY_SPIDERPLOT_HUMAN_CI_PATH, dpi=450, bbox_inches="tight", pad_inches=0.15)
    fig.savefig(POLICY_SPIDERPLOT_HUMAN_CI_SVG_PATH, bbox_inches="tight", pad_inches=0.15)
    plt.show()

print(f"Saved policy spiderplot with human CIs: {POLICY_SPIDERPLOT_HUMAN_CI_PATH}")
print(f"Saved policy spiderplot with human CIs SVG: {POLICY_SPIDERPLOT_HUMAN_CI_SVG_PATH}")
print(f"Saved policy spiderplot human CI table: {POLICY_SPIDERPLOT_HUMAN_CI_TABLE_PATH}")
display(policy_spiderplot_human_ci_table_rounded)


## Paper Metric Table Values

This cell exports the held-out policy comparison table used in the paper. Human entries are reported as mean with percentile bootstrap 95% confidence intervals over held-out participants.

In [ ]:
POLICY_METRIC_TABLE_CSV_PATH = RESULTS_DIR / "policy_metric_table.csv"
POLICY_METRIC_TABLE_TEX_PATH = GENERATED_DIR / "policy_metric_table.tex"
POLICY_METRIC_TABLE_BOOTSTRAPS = 10000
POLICY_METRIC_TABLE_CI_LEVEL = 95
POLICY_METRIC_TABLE_SEED = RANDOM_SEED + 43

POLICY_METRIC_LABELS = {
    "competence_control": "Dexterity",
    "safety": "Safety",
    "awareness": "Social Aware.",
    "positive_impression": "Impression",
}


def metric_table_bootstrap_mean_ci(values, *, n_resamples=POLICY_METRIC_TABLE_BOOTSTRAPS, seed=POLICY_METRIC_TABLE_SEED):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan, 0
    if len(values) == 1:
        value = float(values[0])
        return value, value, value, 1

    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    bootstrap_means = values[sample_indices].mean(axis=1)
    ci_probability = POLICY_METRIC_TABLE_CI_LEVEL / 100
    tail_probability = (1 - ci_probability) / 2
    ci_low, ci_high = np.quantile(bootstrap_means, [tail_probability, 1 - tail_probability])
    return float(values.mean()), float(ci_low), float(ci_high), int(len(values))


policy_metric_human_participant_axis = (
    survey_validation_long[survey_validation_long["axis"].isin(CALIBRATION_AXES)]
    .groupby(["participant_id", "policy", "axis"], dropna=False)
    .agg(score_0_100=("score_norm", lambda values: values.mean() * 100))
    .reset_index()
)

policy_metric_auto_axis = (
    simulation_analysis_long_calibrated[
        simulation_analysis_long_calibrated["axis"].isin(CALIBRATION_AXES)
    ]
    .groupby(["policy", "axis"], dropna=False)
    .agg(score_0_100=("sim_score_calibrated", lambda values: values.mean() * 100))
    .reset_index()
)

policy_metric_human_asimovbm = (
    policy_metric_human_participant_axis
    .groupby(["participant_id", "policy"], dropna=False)
    .agg(score_0_100=("score_0_100", "mean"))
    .reset_index()
)

policy_metric_rows = []
for axis_index, axis in enumerate(CALIBRATION_AXES):
    row = {"Metric": POLICY_METRIC_LABELS[axis], "axis": axis}
    for policy_index, policy in enumerate(["A", "B"]):
        values = policy_metric_human_participant_axis[
            (policy_metric_human_participant_axis["policy"] == policy)
            & (policy_metric_human_participant_axis["axis"] == axis)
        ]["score_0_100"].to_numpy()
        mean, ci_low, ci_high, n = metric_table_bootstrap_mean_ci(
            values,
            seed=POLICY_METRIC_TABLE_SEED + policy_index * 20 + axis_index,
        )
        row[f"Policy {policy} Human Mean"] = mean
        row[f"Policy {policy} Human CI Low"] = ci_low
        row[f"Policy {policy} Human CI High"] = ci_high
        row[f"Policy {policy} Human N"] = n
        row[f"Policy {policy} Auto"] = float(
            policy_metric_auto_axis.loc[
                (policy_metric_auto_axis["policy"] == policy)
                & (policy_metric_auto_axis["axis"] == axis),
                "score_0_100",
            ].iloc[0]
        )

    human_delta = row["Policy A Human Mean"] - row["Policy B Human Mean"]
    auto_delta = row["Policy A Auto"] - row["Policy B Auto"]
    row["Human Delta A-B"] = human_delta
    row["Auto Delta A-B"] = auto_delta
    row["Delta_%"] = max(0.0, 100 - abs(auto_delta - human_delta))
    policy_metric_rows.append(row)

asimov_row = {"Metric": "AsimovBM", "axis": "AsimovBM"}
for policy_index, policy in enumerate(["A", "B"]):
    values = policy_metric_human_asimovbm[
        policy_metric_human_asimovbm["policy"] == policy
    ]["score_0_100"].to_numpy()
    mean, ci_low, ci_high, n = metric_table_bootstrap_mean_ci(
        values,
        seed=POLICY_METRIC_TABLE_SEED + 100 + policy_index,
    )
    asimov_row[f"Policy {policy} Human Mean"] = mean
    asimov_row[f"Policy {policy} Human CI Low"] = ci_low
    asimov_row[f"Policy {policy} Human CI High"] = ci_high
    asimov_row[f"Policy {policy} Human N"] = n
    asimov_row[f"Policy {policy} Auto"] = float(
        policy_metric_auto_axis[policy_metric_auto_axis["policy"] == policy]["score_0_100"].mean()
    )

asimov_row["Human Delta A-B"] = asimov_row["Policy A Human Mean"] - asimov_row["Policy B Human Mean"]
asimov_row["Auto Delta A-B"] = asimov_row["Policy A Auto"] - asimov_row["Policy B Auto"]
asimov_row["Delta_%"] = max(0.0, 100 - abs(asimov_row["Auto Delta A-B"] - asimov_row["Human Delta A-B"]))
policy_metric_rows.append(asimov_row)

policy_metric_table = pd.DataFrame(policy_metric_rows)
for policy in ["A", "B"]:
    policy_metric_table[f"Policy {policy} Human CI Half Width"] = (
        policy_metric_table[f"Policy {policy} Human CI High"]
        - policy_metric_table[f"Policy {policy} Human CI Low"]
    ) / 2
policy_metric_table_rounded = policy_metric_table.copy()
round_columns = [column for column in policy_metric_table_rounded.columns if column not in {"Metric", "axis"}]
policy_metric_table_rounded[round_columns] = policy_metric_table_rounded[round_columns].round(1)
policy_metric_table_rounded.to_csv(POLICY_METRIC_TABLE_CSV_PATH, index=False)


def latex_mean_ci(mean, ci_half_width, *, bold=False):
    value = f"{mean:.1f}\\,$\\pm$\\,{ci_half_width:.1f}"
    return f"\\textbf{{{value}}}" if bold else value


def latex_score(value, *, bold=False):
    formatted = f"{value:.1f}"
    return f"\\textbf{{{formatted}}}" if bold else formatted


latex_lines = [
    r"\begin{table}[!ht]",
    r"\centering",
    r"\caption{Comparison of Automated $A$ and Human $H$ Evaluation of the Perceived Metrics between Policy A and B, with $A, H \in [0, 100]$. Human entries report mean $\pm$ 95\% CI half-width.}",
    r"\label{tab:metrics}",
    r"\footnotesize",
    r"\begin{tabular}{lccccc}",
    r"\toprule",
    r"& \multicolumn{2}{c}{\textbf{Policy A}} & \multicolumn{2}{c}{\textbf{Policy B}} & \textbf{Alignment} \\",
    r"\cmidrule(lr){2-3} \cmidrule(lr){4-5}",
    r"\textbf{Metric} & {Human} & {Auto} & {Human} & {Auto} & {$\Delta_{\%}$} \\",
    r"\midrule",
]

for row_index, row in policy_metric_table.iterrows():
    is_summary = row["Metric"] == "AsimovBM"
    if is_summary:
        latex_lines.append(r"\midrule")
    metric_label = r"\textbf{AsimovBM}" if is_summary else row["Metric"]
    latex_lines.append(
        " & ".join(
            [
                metric_label,
                latex_mean_ci(row["Policy A Human Mean"], row["Policy A Human CI Half Width"], bold=is_summary),
                latex_score(row["Policy A Auto"], bold=is_summary),
                latex_mean_ci(row["Policy B Human Mean"], row["Policy B Human CI Half Width"], bold=is_summary),
                latex_score(row["Policy B Auto"], bold=is_summary),
                latex_score(row["Delta_%"], bold=is_summary),
            ]
        )
        + r" \\")

latex_lines.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\begin{flushleft}",
        r"\scriptsize Human uncertainty is reported as the half-width of the percentile bootstrap 95\% confidence interval over held-out participants. $\Delta_{\%}=100-\left|\left(A_{\mathrm{Policy\ A}}-A_{\mathrm{Policy\ B}}\right)-\left(H_{\mathrm{Policy\ A}}-H_{\mathrm{Policy\ B}}\right)\right|$ compares the automated and human Policy A--Policy B gaps on the $[0,100]$ scale.",
        r"\end{flushleft}",
        r"\end{table}",
    ]
)

policy_metric_latex = "\n".join(latex_lines)
POLICY_METRIC_TABLE_TEX_PATH.write_text(policy_metric_latex)

print(f"Saved policy metric table CSV: {POLICY_METRIC_TABLE_CSV_PATH}")
print(f"Saved policy metric table LaTeX: {POLICY_METRIC_TABLE_TEX_PATH}")
print(policy_metric_latex)
display(policy_metric_table_rounded)


### How To Interpret The Output

A useful benchmark should usually satisfy these checks:

- The survey paired differences are stable enough to interpret, ideally with bootstrap intervals that do not cross zero for the main comparisons.
- The simulation deltas have the same direction as the survey deltas for the same policy/POV comparison.
- The rank order of Q1, Q2, Q3, and Q4 is similar between survey and simulation.
- The simulation has valid runs for every condition and episode slot.

Important limits:

- The current survey order was fixed, so order effects are possible.
- The simulation side has episode/run variability, not participant variability.
- If the simulation scores are saturated near `1.0`, the benchmark may not be calibrated even when it ranks correctly.
- With only three episode slots, simulation-side statistical claims should stay descriptive.

So the practical conclusion is: if the direction agreement and rank alignment are high, the benchmark is directionally useful. If they are low, the metric model or weighting is probably not matching human perception yet.

**What this means**:
The results of the survey were enoisy for the safety and awareness axis. They are both centered close to 0 and have relatively high standard deviation.
The computed pvalues (H0: A-B=0) support this hypothesis, but also suggest that it may be worth working on the concept of "social awareness" a little more, to better define what it does.
The result may actually be expected in this case, since both policy A and policy B are NOT made to be socially aware. There may also be strong evidence that no difference in safety was observed in both cases.

The other 2 metrics appear to be pretty solid

## Result summary
The survey shows a stable human preference for Policy A over Policy B
on competence/control and positive impression.

The calibrated testbench reproduces the same A > B direction
on both stable axes.

For safety and awareness, the survey itself is inconclusive,
so disagreement or agreement there should not be treated as a strong success/failure.